# Training and calibration review

Compare the four targets from the latest completed run. Winner-model promotion should be judged primarily on temporal log loss, Brier score, ECE, probability-sum invariants, and cohort stability—not raw accuracy.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
latest = json.loads((ROOT / "reports/lol/training/latest.json").read_text())
run = Path(latest['report_directory'])
if not run.is_absolute():
    run = ROOT / run
summary = json.loads((run / 'summary.json').read_text())
print(f"Run: {summary['run_id']} | retune={summary['retune']} | Optuna={summary['optuna_allowed']}")

In [ ]:
rows = []
for model in summary['models']:
    flat = pd.json_normalize(model['metrics'], sep='.').iloc[0].to_dict()
    rows.append({'model': model['model_name'], 'target': model['target'], **flat})
metrics = pd.DataFrame(rows).set_index('model')
display(metrics.T)
winner = metrics.loc[[idx for idx in metrics.index if idx.startswith('OutcomePrediction')]]
display(winner.filter(regex='brier|log_loss|ece|probability_sum|calibration', axis=1).T)

In [ ]:
for model in summary['models']:
    calibration = run / model['model_name'] / 'calibration_table.parquet'
    if calibration.exists():
        table = pd.read_parquet(calibration)
        ax = table.plot(x='bin_mean_p', y='bin_emp_rate', marker='o', label=model['target'])
        ax.plot([0, 1], [0, 1], '--', color='grey')
        ax.set(xlabel='predicted probability', ylabel='empirical frequency', title=model['model_name'])